# ◈ Loan Intelligence — Final Notebook Contract

This notebook is aligned with the final `final-loan-2.ipynb` research workflow.

**Pipeline:** cleaning → five engineered features → LabelEncoder → stratified 80/20 split → SMOTE → StandardScaler → nine-model benchmark → XGBoost diagnostics → feature importance → hyperparameter-search experiment.

In [ ]:
!pip -q install -r requirements.txt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import xgboost as xgb

## 1. Dataset and cleaning

Final dataset contract: **45,000 rows × 14 columns**.

Cleaning keeps `person_age <= 80` and `person_emp_exp <= 50`, leaving **44,988 rows**.

In [ ]:
df = pd.read_csv("loan_data.csv")
df_clean = df.dropna(subset=["loan_status"]).copy()
df_clean = df_clean[(df_clean["person_age"] <= 80) & (df_clean["person_emp_exp"] <= 50)].copy()

print("Raw:", df.shape)
print("Clean:", df_clean.shape)

## 2. Feature engineering

The final notebook adds five features:

- `income_to_loan_ratio`
- `emp_length_to_age_ratio`
- `cred_hist_to_age_ratio`
- `high_risk_income_ratio`
- `is_renting`

The final model representation contains **18 features**.

In [ ]:
target = "loan_status"

df_clean["income_to_loan_ratio"] = df_clean["person_income"] / (df_clean["loan_amnt"] + 1)
df_clean["emp_length_to_age_ratio"] = df_clean["person_emp_exp"] / (df_clean["person_age"] + 1)
df_clean["cred_hist_to_age_ratio"] = df_clean["cb_person_cred_hist_length"] / (df_clean["person_age"] + 1)
df_clean["high_risk_income_ratio"] = df_clean["loan_percent_income"] * df_clean["loan_int_rate"]
df_clean["is_renting"] = (df_clean["person_home_ownership"] == "RENT").astype(int)

X = df_clean.drop(columns=[target])
y = df_clean[target].astype(int)

for col in X.select_dtypes(include=["object"]).columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_smote)
X_test_scaled = scaler.transform(X_test)

print("Features:", X.shape[1])
print("Train:", len(X_train), "Test:", len(X_test), "SMOTE train:", len(X_train_smote))

## 3. Nine-model benchmark

The final notebook reports these held-out results on the same test split:

In [ ]:
benchmark = pd.DataFrame([
    ["HistGradientBoosting", 0.932874, 0.860537, 0.8330, 0.846545, 0.976527],
    ["XGBoost", 0.929651, 0.855434, 0.8225, 0.838644, 0.975957],
    ["Random Forest", 0.911202, 0.773079, 0.8500, 0.809717, 0.968021],
    ["Extra Trees", 0.897755, 0.738305, 0.8365, 0.784341, 0.963371],
    ["Gradient Boosting", 0.893532, 0.717809, 0.8585, 0.781876, 0.962550],
    ["SVM", 0.877639, 0.671630, 0.8795, 0.761637, 0.950739],
    ["Decision Tree", 0.880862, 0.703152, 0.8030, 0.749767, 0.853058],
    ["Logistic Regression", 0.861747, 0.635971, 0.8840, 0.739749, 0.944234],
    ["KNN", 0.862192, 0.643396, 0.8525, 0.733333, 0.926571],
], columns=["Model","Accuracy","Precision","Recall","F1","ROC-AUC"])
display(benchmark.sort_values("F1", ascending=False))

## 4. Deployable XGBoost diagnostics

Final notebook XGBoost result:

- Accuracy: **92.9651%**
- Precision: **0.855434**
- Recall: **0.8225**
- F1: **0.838644**
- ROC-AUC: **0.975957**

Confusion matrix: `[[6720, 278], [355, 1645]]`.

In [ ]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
xgb_model.fit(X_train_smote, y_train_smote)
pred = xgb_model.predict(X_test)
prob = xgb_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print(confusion_matrix(y_test, pred))

## 5. Top XGBoost features

The final notebook's global feature-importance chart shows the leading features below. Values are rounded for presentation.

In [ ]:
top_features = pd.Series({
    "previous_loan_defaults_on_file": 88.0,
    "high_risk_income_ratio": 1.9,
    "cb_person_cred_hist_length": 1.4,
    "loan_int_rate": 1.0,
    "person_home_ownership": 0.8,
    "person_education": 0.7,
    "loan_intent": 0.7,
    "person_gender": 0.6,
}).sort_values(ascending=False)

display(top_features.rename("Relative Importance (%)").to_frame())

## 6. Separate XGBoost hyperparameter-search experiment

The notebook's RandomizedSearchCV experiment reports:

```text
n_estimators=250
max_depth=12
learning_rate=0.01
subsample=0.60
colsample_bytree=0.80
gamma=0.00
```

Reported test classification result: approximately **91% accuracy**, **0.77 precision**, **0.86 recall**, **0.81 F1**.

## 7. Interpretation

HistGradientBoosting is the final notebook's benchmark leader. XGBoost remains the repository's deployable model because the application and inference contract are built around it.

Feature importance is model attribution, not causation, and the probability output is not calibrated real-world approval odds.